[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bigdata-com/bigdata-cookbook/blob/main/API_Tutorials/Document_Download_API/document_download.ipynb)

## Install dependencies

Run the cell below **once** to install the required Python packages. You can skip this step if you already have them installed.

In [ ]:
# Install required packages (run this cell once)
%pip install -q requests
print("✓ Required packages installed.")

# Fetch Document API - Usage Guide

**Bigdata.com API** · [`GET /v1/documents/{document_id}`](https://docs.bigdata.com/api-reference/search/fetch-document)

This notebook demonstrates how to fetch entire documents from the Bigdata.com API using the **Fetch Document** endpoint.

---

## Overview

The Fetch Document endpoint returns a time-limited URL to download the document in annotated (structured) JSON format. The downloaded document includes:
- **Document metadata** – source, timestamps, file info
- **Structured content** – title, body blocks (TEXT, TABLE, LIST_ORDERED, LIST_UNORDERED, HEADING, FOOTER)
- **Entity annotations** – companies, people, places, products detected in text
- **Sentence segmentation** – with sentiment scores
- **Analytics** – document-level metrics, detected events, and entity-level analytics
- **Profiling** – processor timestamps

## Response Handling

The endpoint always returns a JSON object with two fields: `url` (a pre-signed URL, valid ~24 hours) and `web_content` (boolean). You always follow the `url` with a second GET request to download the full annotated document JSON.

| `web_content` | Meaning | Difference in downloaded JSON |
|---|---|---|
| `true` | Publicly accessible content (e.g. news articles) | `document.metadata.url` contains a link to the original web page |
| `false` | Premium / non-web content | `document.metadata` has no `url` field |


## Setup

### Prerequisites
- Python 3.8+
- `requests` library
- Valid Bigdata.com API key (set as `BIGDATA_API_KEY` environment variable)

You can generate an API Key at [platform.bigdata.com/api-keys](https://platform.bigdata.com/api-keys)


In [ ]:
import os
import json
import requests

# Verify API key is available
API_KEY = os.getenv('BIGDATA_API_KEY')
if not API_KEY:
    raise ValueError(
        "BIGDATA_API_KEY not found in environment variables. "
        "Please set it before running this notebook."
    )

print("✓ API key loaded successfully")


## Fetch Document Function

The function below calls `GET /v1/documents/{document_id}`, then follows the returned pre-signed URL to download the full annotated document JSON. The `web_content` flag is stored alongside the data for reference.


In [ ]:
def fetch_document(document_id: str) -> dict:
    """
    Fetch a document from the Bigdata.com API.
    
    Calls GET /v1/documents/{document_id} which returns {url, web_content}.
    The url is always a pre-signed URL (~24h validity). A second GET to that
    URL downloads the full annotated document JSON (document, content,
    profiling, analytics).
    
    The web_content flag indicates whether the original source is publicly
    accessible web content. When true, document.metadata.url contains a
    link to the original web page.
    
    Args:
        document_id: 32-character MD5 hex identifier
            (e.g. '776769957735667D2F01F695EF4F1231')
    
    Returns:
        dict with keys:
            - web_content (bool): whether the source is public web content
            - data (dict): full annotated document JSON
    """
    api_key = os.getenv('BIGDATA_API_KEY')
    if not api_key:
        raise ValueError("BIGDATA_API_KEY not found in environment variables")
    
    url = f'https://api.bigdata.com/v1/documents/{document_id}'
    headers = {'X-API-KEY': api_key}
    
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    
    result = response.json()
    web_content = result.get('web_content', False)
    presigned_url = result['url']
    
    doc_response = requests.get(presigned_url)
    doc_response.raise_for_status()
    
    return {
        'web_content': web_content,
        'data': doc_response.json(),
    }

print("✓ Function defined successfully")


## Usage Example

### Fetch Multiple Documents

Let's fetch several documents and examine their structure.


In [ ]:
# Sample document IDs to download
DOCUMENT_IDS = [
    "2B18CA1F6E77EF1E7B8284903554A7F6",  ## Cisco to Lead Consortium to Help Retrain Workers
    "1CDF23AE378D92563C2042D578C8415A",  ## Wyebot Announces Integration with Cisco Catalyst Center
    "F12636B27CED06126D46B755FD57F5A6",   ## Kellanova to Close 2 Production Facilities
    "6FA3173E4785CFDF02ACA75AA95D2CEC",   ## https://uk.finance.yahoo.com/news/trump-bought-netflix-warner-bros-132120706.html
    "9AF93B7690059F5D54A25741811FA99C",   ## https://finance.yahoo.com/news/2026-corvette-zr1-first-drive-the-king-of-the-hill-is-back-160040128.html
    "CE3C157AE0AF3B971B773BEEBC57DEB3"    ## Alliance article 
]



In [ ]:
documents = {}
for doc_id in DOCUMENT_IDS:
    try:
        print(f"Fetching document: {doc_id}...")
        result = fetch_document(doc_id)
        documents[doc_id] = result
        label = "web content" if result['web_content'] else "premium"
        original_url = result['data'].get('document', {}).get('metadata', {}).get('url', '')
        suffix = f" – {original_url}" if original_url else ""
        print(f"  ✓ {label}{suffix}")
    except requests.exceptions.HTTPError as e:
        print(f"  ✗ HTTP {e.response.status_code}: {e.response.text[:200]}")
    except Exception as e:
        print(f"  ✗ Error: {e}")

print(f"\n✓ Fetched {len(documents)} documents successfully")

### Save Documents to Output Folder

Save all fetched documents to the `output/` folder as JSON.


In [ ]:
from pathlib import Path

def save_document(doc_id: str, data: dict, output_dir: str = "output") -> str:
    """
    Save annotated document JSON to the output folder.
    
    Args:
        doc_id: The document ID (used as filename)
        data: The annotated document dict (from the pre-signed URL)
        output_dir: Output directory path
        
    Returns:
        str: Path to the saved file
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    file_path = output_path / f"{doc_id}.json"
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    
    return str(file_path)


OUTPUT_DIR = "output"
print(f"Saving documents to '{OUTPUT_DIR}/' folder...")
print("-" * 60)

saved_files = []
for doc_id, result in documents.items():
    try:
        file_path = save_document(doc_id, result['data'], OUTPUT_DIR)
        file_size = os.path.getsize(file_path)
        saved_files.append((doc_id, file_path, file_size))
        print(f"✓ {doc_id} -> {file_path} ({file_size:,} bytes)")
    except Exception as e:
        print(f"✗ {doc_id} -> Error: {e}")

print("-" * 60)
print(f"✓ Saved {len(saved_files)} documents to '{OUTPUT_DIR}/' folder")


### Extract Full Text Content

Concatenate all body text blocks into a single readable text.



In [ ]:
def extract_body_text(data: dict) -> str:
    """
    Extract and concatenate all body text from an annotated document.
    
    Args:
        data: The annotated document dict (with 'content' key)
        
    Returns:
        str: Concatenated body text with paragraph breaks
    """
    body = data.get('content', {}).get('body', [])
    texts = [block.get('text', '') for block in body if block.get('text')]
    return '\n\n'.join(texts)


if documents:
    first_doc_id = list(documents.keys())[0]
    data = documents[first_doc_id]['data']
    
    title = data.get('content', {}).get('title', {}).get('text', 'N/A')
    body_text = extract_body_text(data)
    
    print("=" * 80)
    print(f"DOCUMENT: {first_doc_id}")
    print("=" * 80)
    print(f"\nTITLE: {title}\n")
    print("-" * 80)
    print("FULL BODY TEXT:")
    print("-" * 80)
    print(body_text)
else:
    print("No documents available.")


In [ ]:
OUTPUT_DIR_TXT = Path(OUTPUT_DIR)

for doc_id, result in documents.items():
    data = result['data']
    title = data.get('content', {}).get('title', {}).get('text', 'N/A')
    body_text = extract_body_text(data)
    file_path = OUTPUT_DIR_TXT / f"{doc_id}.txt"
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(f"{title}\n\n")
        f.write(body_text)
    print(f"✓ {doc_id} -> {file_path}")

### Summary of All Fetched Documents

Display a summary table of all fetched documents.


In [ ]:
print("=" * 120)
print("FETCHED DOCUMENTS SUMMARY")
print("=" * 120)
print(f"\n{'Document ID':<36} {'web_content':<14} {'Source':<22} {'Title (truncated)':<46}")
print("-" * 120)

for doc_id, result in documents.items():
    data = result['data']
    web = str(result['web_content']).lower()
    source_name = data.get('document', {}).get('source', {}).get('name', 'N/A')[:21]
    title = data.get('content', {}).get('title', {}).get('text', 'N/A')[:45]
    print(f"{doc_id:<36} {web:<14} {source_name:<22} {title}")


### Explore Analytics & Profiling

Documents include an `analytics` section (document-level metrics, detected events, entity-level analytics) and a `profiling` section (processor timestamps).

In [ ]:
if documents:
    first_doc_id = list(documents.keys())[0]
    data = documents[first_doc_id]['data']
    
    analytics = data.get('analytics', {})
    doc_analytics = analytics.get('document', {})
    
    print("=" * 80)
    print(f"ANALYTICS for {first_doc_id}")
    print("=" * 80)
    
    print("\n--- Document-Level Metrics ---")
    for key in ['document_type', 'document_sentiment', 'document_sentiment_confidence',
                'composite_sentiment_score', 'product_key', 'realtime']:
        print(f"  {key}: {doc_analytics.get(key, 'N/A')}")
    
    events = analytics.get('events', [])
    print(f"\n--- Events ({len(events)} detected) ---")
    for evt in events[:5]:
        topic = evt.get('topic', '?')
        evt_type = evt.get('type', '?')
        relevance = evt.get('event_relevance', '?')
        print(f"  [{topic}/{evt_type}] relevance={relevance}")
        for role in evt.get('roles', [])[:2]:
            print(f"    -> {role.get('event_detected_entity_name', '?')}: "
                  f"sentiment={role.get('event_sentiment', '?')}")
    
    entities = analytics.get('entities', [])
    print(f"\n--- Entity Analytics ({len(entities)} entities) ---")
    for ent in entities[:10]:
        print(f"  {ent.get('entity_name', '?'):<30} "
              f"type={ent.get('entity_type', '?'):<6} "
              f"relevance={ent.get('entity_relevance', '?'):<4} "
              f"sentiment={ent.get('entity_sentiment', '?')}")
    
    profiling = data.get('profiling', {})
    collection = profiling.get('collection', {})
    print(f"\n--- Profiling ---")
    print(f"  processor_in:  {collection.get('processor_in_timestamp_utc', 'N/A')}")
    print(f"  processor_out: {collection.get('processor_out_timestamp_utc', 'N/A')}")
else:
    print("No documents to inspect.")